# Product Line Profitability & Margin Performance Analysis
## Nassau Candy Distributor

This notebook contains the Exploratory Data Analysis (EDA), Data Cleaning, and insights for the Nassau Candy Distributor dataset.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

# Set styling
sns.set_theme(style="whitegrid")

## Data Cleaning & Validation
- Validate cost and sales values
- Remove zero-sales or invalid profit records
- Handle missing unit values
- Standardize labels

In [ ]:
# Load the dataset
df = pd.read_csv('Nassau Candy Distributor.csv')

# 1. Validate cost and sales values, remove zero-sales
initial_len = len(df)
df = df[(df['Sales'] > 0) & (df['Cost'] > 0)]

# 2. Handle missing unit values (fill with 1 or drop, here we drop if 0)
df = df[df['Units'] > 0]
df['Units'] = df['Units'].fillna(1)

# 3. Standardize labels
df['Division'] = df['Division'].str.strip().str.title()
df['Product Name'] = df['Product Name'].str.strip()

print(f"Removed {initial_len - len(df)} invalid records.")
df.head()

## Profitability Metric Calculation
- **Gross Margin (%)** = (Gross Profit / Sales) * 100
- **Profit per Unit** = Gross Profit / Units
- **Total Profit Contribution** = Product Profit / Total Profit

In [ ]:
# Calculate Metrics
df['Gross Margin (%)'] = (df['Gross Profit'] / df['Sales']) * 100
df['Profit per Unit'] = df['Gross Profit'] / df['Units']

total_profit = df['Gross Profit'].sum()
df['Profit Contribution (%)'] = (df['Gross Profit'] / total_profit) * 100

df[['Product Name', 'Sales', 'Gross Profit', 'Gross Margin (%)', 'Profit per Unit', 'Profit Contribution (%)']].head()

## Product-Level Profitability Analysis
Rank products by Gross profit and Gross margin. Identify high/low performers.

In [ ]:
product_group = df.groupby('Product Name').agg({
    'Sales': 'sum',
    'Gross Profit': 'sum',
    'Units': 'sum'
}).reset_index()

product_group['Gross Margin (%)'] = (product_group['Gross Profit'] / product_group['Sales']) * 100

# High-profit products
print("Top 5 Products by Gross Profit:")
display(product_group.sort_values(by='Gross Profit', ascending=False).head(5))

# Low-margin products
print("Bottom 5 Products by Gross Margin:")
display(product_group.sort_values(by='Gross Margin (%)', ascending=True).head(5))

## Division-Level Performance Analysis

In [ ]:
division_group = df.groupby('Division').agg({
    'Sales': 'sum',
    'Gross Profit': 'sum'
}).reset_index()
division_group['Gross Margin (%)'] = (division_group['Gross Profit'] / division_group['Sales']) * 100

fig = px.bar(division_group, x='Division', y='Gross Profit', color='Gross Margin (%)',
             title='Division Profitability', text_auto='.2s')
fig.show()

## Profit Concentration (Pareto) Analysis
Determine % of products contributing 80% of revenue and 80% of profit.

In [ ]:
product_group = product_group.sort_values(by='Gross Profit', ascending=False)
product_group['Cumulative Profit %'] = 100 * (product_group['Gross Profit'].cumsum() / product_group['Gross Profit'].sum())

# Products that make up 80% of profit
top_80_profit = product_group[product_group['Cumulative Profit %'] <= 80]
print(f"{len(top_80_profit)} out of {len(product_group)} products contribute to 80% of total profit.")

fig = px.line(product_group, x='Product Name', y='Cumulative Profit %', title='Pareto Chart for Profit')
fig.add_hline(y=80, line_dash="dash", annotation_text="80% Threshold")
fig.show()

## Cost Structure Diagnostics
Cost vs Sales scatter analysis.

In [ ]:
fig = px.scatter(df, x='Sales', y='Cost', color='Division', 
                 hover_data=['Product Name'], title='Cost vs Sales per Order')
fig.show()